In [ ]:
import pandas as pd
import plotly.express as px
import pickle
from IPython.display import VimeoVideo
from scipy.stats.mstats import trimmed_var
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.metrics import silhouette_score
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.utils.validation import check_is_fitted
import warnings

warnings.simplefilter(action="ignore", category=FutureWarning)
warnings.filterwarnings("ignore", category=DeprecationWarning)


In [ ]:
VimeoVideo("714612789", h="f4f8c10683", width=600)


In [ ]:
VimeoVideo("714612746", h="07dc57f72c", width=600)


In [ ]:
def wrangle(filepath):
    # read file into dataFrame
    df = pd.read_csv(filepath)
    mask = (df["TURNFEAR"]==1) & (df["NETWORTH"]<2e6)
    df = df[mask]
    return df


In [ ]:
df = wrangle("data/SCFP2019.csv.gz")

print("df type:", type(df))
print("df shape:", df.shape)
df.head()


In [ ]:
VimeoVideo("714612679", h="040facf6e2", width=600)


In [ ]:
# Calculate variance, get 10 largest features
top_ten_var = df.var().sort_values().tail(10)

print("top_ten_var type:", type(top_ten_var))
print("top_ten_var shape:", top_ten_var.shape)
top_ten_var


In [ ]:
VimeoVideo("714612647", h="5ecf36a0db", width=600)


In [ ]:
# Create horizontal bar chart of `top_ten_var`
fig = px.bar(
    x=top_ten_var,
    y=top_ten_var.index,
    title="SCF: High Variance Features"
)
fig.update_layout(xaxis_title="Variance", yaxis_title="Feature")
fig.show()


In [ ]:
VimeoVideo("714612615", h="9ae23890fc", width=600)


In [ ]:
# Create a boxplot of `NHNFIN`
fig = px.box(
    data_frame=df,
    x= "NHNFIN",
    title = "Distribution of Non-home, Non-Financial Assets"
)

fig.update_layout(xaxis_title="Value [$]")

fig.show()


In [ ]:
VimeoVideo("714612570", h="b1be8fb750", width=600)


In [ ]:
# Calculate trimmed variance
top_ten_trim_var = df.apply(trimmed_var, limits=(0.1,0.1)).sort_values().tail()

print("top_ten_trim_var type:", type(top_ten_trim_var))
print("top_ten_trim_var shape:", top_ten_trim_var.shape)
top_ten_trim_var


In [ ]:
VimeoVideo("714611188", h="d762a98b1e", width=600)


In [ ]:
# Create horizontal bar chart of `top_ten_trim_var`
fig = px.bar(
    x=top_ten_trim_var,
    y=top_ten_trim_var.index,
    title="SCF: High Variance Features"
)

fig.update_layout(xaxis_title="Trimmed Variance", yaxis_title="Feature")
fig.show()


In [ ]:
VimeoVideo("714611161", h="61dee490ee", width=600)


In [ ]:
high_var_cols = top_ten_trim_var.tail(5).index.to_list()

print("high_var_cols type:", type(high_var_cols))
print("high_var_cols len:", len(top_ten_trim_var))
high_var_cols


In [ ]:
VimeoVideo("714611148", h="f7fbd4bcc5", width=600)


In [ ]:
X = df[high_var_cols]

print("X type:", type(X))
print("X shape:", X.shape)
X.head()


In [ ]:
VimeoVideo("714611113", h="3671a603b5", width=600)


In [ ]:
X_summary = X.aggregate(["mean","std"]).astype(int)

print("X_summary type:", type(X_summary))
print("X_summary shape:", X_summary.shape)
X_summary


In [ ]:
VimeoVideo("714611056", h="670f6bdb78", width=600)


In [ ]:
# Instantiate transformer
ss = StandardScaler()

# Transform `X`
X_scaled_data = ss.fit_transform(X)

# Put `X_scaled_data` into DataFrame
X_scaled = pd.DataFrame(X_scaled_data, columns=X.columns)

print("X_scaled type:", type(X_scaled))
print("X_scaled shape:", X_scaled.shape)
X_scaled.head()


In [ ]:
VimeoVideo("714611032", h="1ed03c46eb", width=600)


In [ ]:
X_scaled_summary = X_scaled.aggregate(["mean","std"]).astype(int)

print("X_scaled_summary type:", type(X_scaled_summary))
print("X_scaled_summary shape:", X_scaled_summary.shape)
X_scaled_summary


In [ ]:
VimeoVideo("714610976", h="82f32af967", width=600)


In [ ]:
n_clusters = range(2,13)
inertia_errors = []
silhouette_scores = []

# Add `for` loop to train model and calculate inertia, silhouette score.
for k in n_clusters:
    # build model 
    model = make_pipeline(
        StandardScaler(),
        KMeans(n_clusters=k, random_state=42)
    )
    # train model
    model.fit(X)
    
    # Calculatw inertia
    inertia_errors.append(model.named_steps['kmeans'].inertia_)
    
    # Calculate silhouette_scores
    silhouette_scores.append(
        silhouette_score(X, model.named_steps['kmeans'].labels_)
        )
    
                             
print("inertia_errors type:", type(inertia_errors))
print("inertia_errors len:", len(inertia_errors))
print("Inertia:", inertia_errors)
print()
print("silhouette_scores type:", type(silhouette_scores))
print("silhouette_scores len:", len(silhouette_scores))
print("Silhouette Scores:", silhouette_scores)


In [ ]:
VimeoVideo("714610940", h="bacf42a282", width=600)


In [ ]:
# Create line plot of `inertia_errors` vs `n_clusters`
fig = px.line(
    x= n_clusters, y= inertia_errors, title="K-Means Model: Inertia vs Number of Clusters"
)
fig.update_layout(xaxis_title="Number of Clusters (k)", yaxis_title="Inertia")
fig.show()


In [ ]:
VimeoVideo("714610912", h="01961ee57a", width=600)


In [ ]:
# Create a line plot of `silhouette_scores` vs `n_clusters`
fig = px.line(
    x= n_clusters, y= silhouette_scores, title="K-Means Model: Silhouette Score vs Number of Clusters"
)
fig.update_layout(xaxis_title="Number of Clusters (k)", yaxis_title="Silhouette Score")
fig.show()


In [ ]:
VimeoVideo("714610883", h="a6a0431b02", width=600)


In [ ]:
# Build model
final_model = make_pipeline(
    StandardScaler(),
    KMeans(n_clusters=4, random_state=42)
)

# Fit model to data
final_model.fit(X)


In [ ]:
# Assert that model has been fit to data
check_is_fitted(final_model)


In [ ]:
VimeoVideo("714610862", h="69ff3fb2c8", width=600)


In [ ]:
labels = final_model.named_steps["kmeans"].labels_

print("labels type:", type(labels))
print("labels len:", len(labels))
print(labels[:5])


In [ ]:
VimeoVideo("714610842", h="008a463aca", width=600)


In [ ]:
xgb = X.groupby(labels).mean()

print("xgb type:", type(xgb))
print("xgb shape:", xgb.shape)
xgb


In [ ]:
VimeoVideo("714610772", h="e118407ff1", width=600)


In [ ]:
# Create side-by-side bar chart of `xgb`
fig = px.bar(xgb, barmode="group",
            title="Mean Household Finances by Cluster")
fig.update_layout(xaxis_title="Cluster", yaxis_title="Value [$]")

fig.show()


In [ ]:
VimeoVideo("714610665", h="19c9f7bf7f", width=600)


In [ ]:
# Instantiate transformer
pca = PCA(n_components=2,random_state=42)
# Transform `X`
X_t = pca.fit_transform(X)
# Put `X_t` into DataFrame
X_pca = pd.DataFrame(X_t, columns=["PC1","PC2"])

print("X_pca type:", type(X_pca))
print("X_pca shape:", X_pca.shape)
X_pca.head()


In [ ]:
VimeoVideo("714610491", h="755c66fe15", width=600)


In [ ]:
fig = px.scatter(
    data_frame=X_pca,
    x="PC1",
    y="PC2",
    color=labels.astype(str),
    title="PCA Representation of Clusters"
)
fig.update_layout(xaxis_title="PC1", yaxis_title="PC2")
fig.show()
